Inclusion of libraries:

In [86]:
#Let's add libraries:
import numpy as np
import matplotlib as plt
import scipy as sp
from scipy import *
import ipywidgets as widgets
from ipywidgets import interact
from IPython.display import display
from sympy import symbols, Eq, solve

**Include initial variables**

In [91]:
#Change this for every new material!

style={'description_width':'initial'}
#Make sure that the titles dont get truncated and the titles are important because otherwise ill forget our units
#Ok, redo all I had before but now add it all as sliders/input fields?
MaterialNameInput=widgets.Text(value='', description='Material Name:', style=style)
display(MaterialNameInput)

PermConstInput=widgets.BoundedFloatText(value=3500.0, min=0.0, max=4000.0, step=0.1, description='Perm Const (Barrer):', style=style)
display(PermConstInput)

E_Input=widgets.BoundedFloatText(value=1.5, min=0.0, max=100, step=0.1, description='Young Modulus (MPa), 100%:', style=style)
display(E_Input)

UTS_Input=widgets.BoundedFloatText(value=28, min=0.0, max=100, step=0.1, description='Ultim Tens Stregth (MPa):', style=style) #Remember you will input MPa but must convert to Pa after !
display(UTS_Input)

#Add a slider for moles
n_0_Input=widgets.BoundedFloatText(value=2, min=0.0, max=5.0, step=0.1, description='Moles, n_0 (mol):', style=style)
display(n_0_Input)

T_0_Input=widgets.BoundedFloatText(value=0.5, min=0.0, max=10, step=0.01, description='Thickness, T_0 (mm):', style=style)
display(T_0_Input)

r_0_Input=widgets.BoundedFloatText(value=106, min=0.0, max=1000, step=0.1, description='Radius, r_0 (mm):', style=style)
display(r_0_Input)

rho_input=widgets.BoundedFloatText(value=0.9, min=0.0, max=100, step=0.1, description='Density (g/cm^3):', style=style)
display(rho_input)

cost_input=widgets.BoundedFloatText(value=1.6, min=0.0, max=100, step=0.1, description='Cost ($/kg):', style=style)
display(cost_input)

Shore_hardness_Input=widgets.BoundedFloatText(value=50, min=0.0, max=100, step=0.1, description='Hardness (Shore, A-Level):', style=style)
display(Shore_hardness_Input)

Temperature_Input=widgets.BoundedFloatText(value=25, min=0.0, max=100, step=0.1, description='Temp (C):', style=style)
display(Temperature_Input)

Text(value='', description='Material Name:', style=DescriptionStyle(description_width='initial'))

BoundedFloatText(value=3500.0, description='Perm Const (Barrer):', max=4000.0, step=0.1, style=DescriptionStyl…

BoundedFloatText(value=1.5, description='Young Modulus (MPa), 100%:', step=0.1, style=DescriptionStyle(descrip…

BoundedFloatText(value=28.0, description='Ultim Tens Stregth (MPa):', step=0.1, style=DescriptionStyle(descrip…

BoundedFloatText(value=2.0, description='Moles, n_0 (mol):', max=5.0, step=0.1, style=DescriptionStyle(descrip…

...

BoundedFloatText(value=106.0, description='Radius, r_0 (mm):', max=1000.0, step=0.1, style=DescriptionStyle(de…

BoundedFloatText(value=0.9, description='Density (g/cm^3):', step=0.1, style=DescriptionStyle(description_widt…

BoundedFloatText(value=1.6, description='Cost ($/kg):', step=0.1, style=DescriptionStyle(description_width='in…

BoundedFloatText(value=50.0, description='Hardness (Shore, A-Level):', step=0.1, style=DescriptionStyle(descri…

BoundedFloatText(value=25.0, description='Temp (C):', step=0.1, style=DescriptionStyle(description_width='init…

In [97]:
#Now, cook up some initial values from all the kerfuffle up top:

#Oh and remember to convert all units!

r_0=r_0_Input.value*1e-3 #Converting mm to m
T_0=T_0_Input.value*1e-3 #Converting mm to m
rho=rho_input.value*1e3 #Converting g/cm^3 to kg/m^3
n_0=n_0_Input.value
E=E_Input.value*1e6 #Converting MPa to Pa
UTS=UTS_Input.value*1e6 #Converting MPa to Pa
PermConst=PermConstInput.value*7.5005e-18 #Converting Barrer to m^2 s^-1 Pa^-1 (stupid ahh units)
cost=cost_input.value
Shore_hardness=Shore_hardness_Input.value
MaterialName=MaterialNameInput.value
Temp=Temperature_Input.value+273.15 #Convert temp to K

pi=np.pi #I would go crazy writing np.pi so I'm just replacing it

p_atm=101325 #Atmospheric pressure in Pa

R=8.3144598 #Gas constant in J/mol/K

V_stretchless=(4/3)*pi*r_0**3

print(f"The normalised units are: r_0={r_0} m, T_0={T_0} m, \nrho={rho} kg/m^3, n_0={n_0} mol, E={E} Pa, UTS={UTS} Pa, \nPermConst={PermConst} m^2 s^-1 Pa^-1, cost={cost} $/kg, Shore_hardness={Shore_hardness}. Temp is {Temp} K")

print(f"The initial spherical, unstretched volume is then V_0={V_stretchless} m^3")

#Note that it's actually really fascinating how sensitive V_stretchless is to r_0. Of course, it's a cubic relationship, so this is expected, but it's still interesting to see!

The normalised units are: r_0=0.106 m, T_0=0.001 m, 
rho=900.0 kg/m^3, n_0=2.0 mol, E=1500000.0 Pa, UTS=28000000.0 Pa, 
PermConst=2.625175e-14 m^2 s^-1 Pa^-1, cost=1.6 $/kg, Shore_hardness=50.0. Temp is 298.15 K
The initial spherical, unstretched volume is then V_0=0.004988916154543867 m^3


In [85]:
#Then, designate all time-evolvable variables with their initial values
"""r=r_0
T=T_0
n=n_0
V=V_0"""

#Maybe it'll be better to leave it for now and then redefine it as a looping function? Idk

'r=r_0\nT=T_0\nn=n_0\nV=V_0'

**Add hard limitations for model**

In [95]:
V_lim=0.0048 #If volume is <4.8L then the bladder is too deflated to provide enough buoyancy.

**Calculate Initial Variables**

In [96]:
#Calculate packed material mass from the area of the material initially, taken as a sphere with r_0 radius, using thickness and density to find mass of material.
SA=4*pi*r_0**2 #Surface area of our material
V_material=SA*T_0 #Volume of our material
MaterialMass=V_material*rho
MaterialCost=MaterialMass*cost
print(f"The initial mass of the packed material is {MaterialMass:.4f} kg, or {MaterialMass*1e3:.1f} g")
print(f"The cost of the packed material is ${MaterialCost:.4f}")

The initial mass of the packed material is 0.1271 kg, or 127.1 g
The cost of the packed material is $0.2033


**Start the real deal now** - Do everything for t=0.

In [99]:
#We know that our pressures must be the same from inside and from the atmosphere+bladder's restorative pressure
#So, we want to solve f(r)=0=p_gas(r,n)-(p_atm+p_res(r))

r=symbols('r') #Define radius as a movable variable? Man idk what im about to start cooking, wish me luck

#From our calculations, we know that:
p_res=(2*E*(r_0**2)*T_0*(r-r_0))/(r**3)

#We also know that:
p_gas=(3*n_0*R*Temp)/(4*pi*r**2)

InitialPressureEqualities=Eq(p_gas-p_res-p_atm,0)

InitialPressureRadiusSolved=solve(InitialPressureEqualities,r)
print(InitialPressureRadiusSolved)

#def VolumeEvolution():


[-0.104941246987614 + 0.e-22*I, -0.00310990147477502 + 0.e-24*I, 0.108051148462389 - 0.e-22*I]
